# AgriSight: Colab quickstart

Runtime → Change runtime type → **T4 GPU**. Runs data prep, training and evaluation of the vision model. Results are saved to Google Drive so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Replace with your GitHub repo URL
!git clone https://github.com/<you>/agrisight.git
%cd agrisight
!pip install -q timm scikit-learn sentence-transformers rank-bm25 pypdf onnx onnxruntime
!pip install -q -e .

## 1. Data
PlantVillage is large. If the git clone is slow, download the Kaggle copy instead and point `--plantvillage` at its `color/` folder.

In [ ]:
!bash scripts/download_data.sh
!python scripts/prepare_data.py --plantvillage data/raw/PlantVillage-Dataset/raw/color --plantdoc data/raw/PlantDoc-Dataset

## 2. Train
Save runs to Drive. Use `--limit 2000 --epochs 2` first to check everything works.

In [ ]:
RUNS = '/content/drive/MyDrive/agrisight/runs'
!sed -i "s#runs_dir: runs#runs_dir: {RUNS}#" configs/default.yaml
!python -m agrisight.train --run-name convnext_field_aug --epochs 10

## 3. Evaluate on lab and field photos

In [ ]:
!python -m agrisight.evaluate --ckpt {RUNS}/convnext_field_aug/best.pt --save-temperature

In [ ]:
from IPython.display import Image, display
display(Image(f'{RUNS}/convnext_field_aug/confusion_field_test.png', width=700))

## 4. Try a single image

In [ ]:
from PIL import Image as PILImage
from agrisight.predict import Predictor
import csv
p = Predictor(f'{RUNS}/convnext_field_aug/best.pt')
row = next(csv.DictReader(open('data/manifests/field_test.csv')))
print('true:', row['label'])
p.predict(PILImage.open(row['path']))

## Next
- Other backbones and AdaBN/DANN: see `docs/EXPERIMENTS.md`
- For the VLM on Colab set `vlm.backend: hf` in `configs/default.yaml` (Qwen2.5-VL-3B fits on a T4 in fp16)